# Train spatial/non-spatial classifier

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
SEED = 42
np.random.seed(SEED)

## ✂ Split our manually labelled dataset into train/val/test (80/10/10)

In [7]:
data = (
    pd.read_excel('interim/is_geospatial_20260428_edits.xlsx')
    .dropna(subset=['is_geospatial_human_check'])
    .assign(
        y=lambda df_: df_.is_geospatial_human_check.astype(int)
    )
    .filter(['query_id', 'query_text', 'y'])
)

print(data.y.value_counts())

data

y
0    632
1    568
Name: count, dtype: int64


,query_id,query_text,y
0,891609,what river runs through southern california,1
1,848519,what is the statue of limitation to collect a ...,0
2,485574,reactive arthritis from salmonella,0
3,375479,how to register customary land in png,1
4,314185,how much does insulin cost for humans,0
...,...,...,...
4976,1152306,what is the adr,0
4981,848484,what is the state tree for washington,1
4983,945399,when do you renew your massachusetts driver's ...,1
4986,543451,"weather in beaumont, california fahrenheit",1


In [15]:
# First split: hold out 800 for test
train_val_df, test_df = train_test_split(
    data,
    test_size=800,
    random_state=SEED,
    stratify=data['y'] # ensure equal number of spatial/non-spatial
)

# Second split: from the remaining 400, take 200 for val
train_df, val_df = train_test_split(
    train_val_df,
    test_size=200,
    random_state=SEED,
    stratify=train_val_df['y']
)

In [16]:
assert len(train_df) + len(val_df) + len(test_df) == len(data)

print('==Train counts==')
print(train_df.y.value_counts())

print('\n\n==Val counts==')
print(val_df.y.value_counts())

print('\n\n==Test counts==')
print(test_df.y.value_counts())

==Train counts==
y
0    105
1     95
Name: count, dtype: int64


==Val counts==
y
0    106
1     94
Name: count, dtype: int64


==Test counts==
y
0    421
1    379
Name: count, dtype: int64


In [17]:
train_df.to_csv('output/is_geospatial.train.csv', index=False)
val_df.to_csv('output/is_geospatial.val.csv', index=False)
test_df.to_csv('output/is_geospatial.test.csv', index=False)

# 💪 Train our classifier

In [18]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
import torch

/Users/ilya/miniconda3/lib/python3.11/site-packages/gunicorn/util.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [19]:
train_ds = Dataset.from_pandas(
    pd.read_csv('output/is_geospatial.train.csv', usecols=['query_text', 'y'])
)

val_ds = Dataset.from_pandas(
    pd.read_csv('output/is_geospatial.val.csv', usecols=['query_text', 'y'])
)

test_ds  = Dataset.from_pandas(
    pd.read_csv('output/is_geospatial.test.csv', usecols=['query_text', 'y'])
)

In [20]:
model = SetFitModel.from_pretrained('BAAI/bge-small-en-v1.5')
# warning of initialising classification head with random weights is OK and expected!

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [27]:
args = TrainingArguments(
    num_epochs=5,
    num_iterations=20,
    batch_size=64,
    body_learning_rate=2e-5,
    seed=SEED,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='embedding_loss',
    greater_is_better=True
)

def compute_metrics(y_pred, y_true):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    column_mapping={'query_text': 'text', 'y': 'label'},
    metric=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# disable pin_memory in the internal HF trainer
trainer.st_trainer.args.dataloader_pin_memory = False

Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [28]:
trainer.train()

***** Running training *****
  Num unique pairs = 8000
  Batch size = 64
  Num epochs = 5


Epoch,Training Loss,Validation Loss
1,0.000600,0.070221
2,0.000300,0.067414
3,0.000300,0.062241


In [29]:
results = trainer.evaluate(test_ds)
print(f"Accuracy: {results['accuracy']:.3f}")
print(f"F1: {results['f1']:.3f}")

Applying column mapping to the evaluation dataset
***** Running evaluation *****


Accuracy: 0.934
F1: 0.931


In [31]:
test_texts = test_df['query_text'].tolist()
test_labels = test_df['y'].tolist()
preds = model.predict(test_texts)

# Build a results dataframe
results_df = pd.DataFrame({
    'text': test_texts,
    'true': test_labels,
    'pred': preds,
})

# Convert preds to int if they come back as tensors/strings
results_df['pred'] = results_df['pred'].astype(int)
results_df['true'] = results_df['true'].astype(int)

# Filter to errors
errors = results_df[results_df['true'] != results_df['pred']].copy()
errors['error_type'] = errors.apply(
    lambda r: 'false_positive' if r['pred'] == 1 else 'false_negative',
    axis=1,
)

print(f'Total errors: {len(errors)} / {len(results_df)}')
print(errors['error_type'].value_counts())

Total errors: 53 / 800
error_type
false_positive    31
false_negative    22
Name: count, dtype: int64


In [32]:
# False positives - model said 1, truth is 0
fp = errors[errors['error_type'] == 'false_positive']
print(f'\n=== False positives ({len(fp)}) ===')
for _, row in fp.head(20).iterrows():
    print(f'- {row["text"]}')

# False negatives - model said 0, truth is 1
fn = errors[errors['error_type'] == 'false_negative']
print(f'\n=== False negatives ({len(fn)}) ===')
for _, row in fn.head(20).iterrows():
    print(f'- {row["text"]}')


=== False positives (31) ===
- when is the good time to see northern lights
- what is microsoft al
- what year was the original aladdin made
- trump what nationality
- est uber cost
- definition bray
- what is the length of earth days on saturn
- state license board number for contractors
- what is bardo
- what is the name of the us armed forces telephone network
- what conference is it notre dame basketball on
- what all places do i need to change my address when i move
- when are the peak seasons prices at disneyland
- where does cantonese food come from
- what time is mid shift at southern wine and spirits
- biggest snakes in world
- what is in dcd
- when did the gulf of tonkin incident happen
- how many teams are in the eastern conference
- where is area code

=== False negatives (22) ===
- how much do granite fabricators pay for granite
- how many steps in the arc de triomphe
- who's statue is at the top of little round top
- how wide is the base of the great wall
- what language

In [38]:
trainer.model.save_pretrained('geospatial_query_classifier_eval')

### Let's stress-test it (sanity/sense check)

In [39]:
model = SetFitModel.from_pretrained('geospatial_query_classifier_eval')

In [35]:
def is_spatial(q):
    pred = model.predict([q])
    print(f'{q}: {"✅" if pred[0] else "🚫"} \n---')

In [36]:
is_spatial('is it a long flight london to kaunas')

is it a long flight london to kaunas: ✅ 
---


In [37]:
# clearly spatial
is_spatial('cafes within walking distance')
is_spatial('nearest hospital')
is_spatial('distance from paris to berlin')
is_spatial('where am i right now')
is_spatial('countries bordering ukraine')
is_spatial('borderline invisible')
is_spatial('elevation of mount fuji')
is_spatial('how far is the nearest bus stop')
is_spatial('route from oxford to cambridge')
is_spatial('restaurants along the m1')
is_spatial('stop it m8')
is_spatial('flood risk in this area')

# clearly non-spatial (metaphorical / abstract)
is_spatial('close to my heart')
is_spatial('far from the truth')
is_spatial('a long way to go in my career')
is_spatial('high-level overview')
is_spatial('deep learning basics')
is_spatial('near impossible')
is_spatial('where do i stand politically')
is_spatial('on the edge emotionally')

# ambiguous / borderline (good test cases)
is_spatial('where should i live')              # needs disambiguation
is_spatial('how far can i go with this idea')  # metaphorical by default
is_spatial('where should i invest my money')   # abstract 'where'
is_spatial('how close are we to a solution')
is_spatial('what is my position on this')
is_spatial('where does this leave us')
is_spatial('how far apart are the classes')

# mixed spatial + non-spatial intent
is_spatial('how far is too far in relationships')
is_spatial('close friends who live far away')
is_spatial('where can i escape mentally')
is_spatial('distance learning programmes near me')

# tricky linguistic traps
is_spatial('where do i belong')
is_spatial('long way home')
is_spatial('keep your distance')
is_spatial('at a crossroads in life')
is_spatial('moving forward with the plan')

cafes within walking distance: ✅ 
---
nearest hospital: ✅ 
---
distance from paris to berlin: ✅ 
---
where am i right now: ✅ 
---
countries bordering ukraine: ✅ 
---
borderline invisible: 🚫 
---
elevation of mount fuji: ✅ 
---
how far is the nearest bus stop: ✅ 
---
route from oxford to cambridge: ✅ 
---
restaurants along the m1: ✅ 
---
stop it m8: 🚫 
---
flood risk in this area: ✅ 
---
close to my heart: 🚫 
---
far from the truth: 🚫 
---
a long way to go in my career: 🚫 
---
high-level overview: 🚫 
---
deep learning basics: 🚫 
---
near impossible: 🚫 
---
where do i stand politically: 🚫 
---
on the edge emotionally: 🚫 
---
where should i live: ✅ 
---
how far can i go with this idea: 🚫 
---
where should i invest my money: 🚫 
---
how close are we to a solution: 🚫 
---
what is my position on this: 🚫 
---
where does this leave us: ✅ 
---
how far apart are the classes: 🚫 
---
how far is too far in relationships: 🚫 
---
close friends who live far away: ✅ 
---
where can i escape mentally: 🚫 


## Let's now train on the whole labelled set (1,200)

Before we do our full pass on 1M queries

In [40]:
from datasets import concatenate_datasets
from setfit import SetFitModel, Trainer, TrainingArguments

In [41]:
full_ds = concatenate_datasets([train_ds, val_ds, test_ds])

In [48]:
pd.Series(full_ds['y']).value_counts()

0    632
1    568
Name: count, dtype: int64

In [42]:
model_prod = SetFitModel.from_pretrained('BAAI/bge-small-en-v1.5')

args_prod = TrainingArguments(
    num_epochs=3, # training stopped after 3 epochs so let's do 3 epochs.
    num_iterations=20,
    batch_size=64,
    body_learning_rate=2e-5,
    seed=SEED,
    logging_steps=50,
    eval_strategy='no',
    save_strategy='no',
    load_best_model_at_end=False,
)

trainer_prod = Trainer(
    model=model_prod,
    args=args_prod, # using same args as above except for eval strategy
    train_dataset=full_ds,
    eval_dataset=None,  # no validation - we are not tuning this time!
    column_mapping={'query_text': 'text', 'y': 'label'},
)

trainer_prod.st_trainer.args.dataloader_pin_memory = False
trainer_prod.train()

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Applying column mapping to the training dataset


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 48000
  Batch size = 64
  Num epochs = 3


Step,Training Loss
1,0.237400
50,0.238300
100,0.215100
150,0.098200
200,0.034000
250,0.010400
300,0.005600
350,0.003500
400,0.002000
450,0.001300


In [43]:
# Save locally (just in case)
trainer_prod.model.save_pretrained('is-geospatial-query')

# AND

# Upload directly to the Huggingface Hub
trainer_prod.model.push_to_hub(
    'ilyankou/is-geospatial-query',
    commit_message='Upload v1.0',
    private=False
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...-geospatial-query/model.safetensors:   0%|          |  554kB /  133MB            

  .../is-geospatial-query/model_head.pkl:   1%|          |  32.0B / 3.94kB            

CommitInfo(commit_url='https://huggingface.co/ilyankou/is-geospatial-query/commit/c9bfb81466333e5961f84a86485e48cca7fff6eb', commit_message='Upload v1.0', commit_description='', oid='c9bfb81466333e5961f84a86485e48cca7fff6eb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ilyankou/is-geospatial-query', endpoint='https://huggingface.co', repo_type='model', repo_id='ilyankou/is-geospatial-query'), pr_revision=None, pr_num=None)

# Classify *all* MS MARCO queries

- Runs about 10 min (on an M3 processor)

In [49]:
from setfit import SetFitModel
import pandas as pd
from tqdm import tqdm

In [51]:
# Load the model
spatial_classifier = SetFitModel.from_pretrained("ilyankou/is-geospatial-query")

# Read queries
queries = pd.read_csv("interim/queries.csv.zip")

# Define batch prediction function
def batch_predict(texts, batch_size=1024):
    preds = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        preds.extend(p.item() for p in spatial_classifier.predict(batch))
    return preds #.item()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config_setfit.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model_head.pkl:   0%|          | 0.00/3.94k [00:00<?, ?B/s]

In [52]:
def ask_geospatial_detector(q):
    pred = spatial_classifier.predict([q])
    print(f'{q}: {"✅" if pred[0] else "🚫"} \n---')

ask_geospatial_detector('countries bordering ukraine')
ask_geospatial_detector('elevation of mount fuji')
ask_geospatial_detector('who when and where')
ask_geospatial_detector('where are they hiding')

countries bordering ukraine: ✅ 
---
elevation of mount fuji: ✅ 
---
who when and where: 🚫 
---
where are they hiding: ✅ 
---


In [53]:
# Run prediction
queries['is_geospatial_pred'] = batch_predict( queries['query_text'].tolist() )

100%|█████████████████████████████████████████| 988/988 [10:52<00:00,  1.51it/s]


In [54]:
queries.groupby('is_geospatial_pred', group_keys=False).sample(n=5, random_state=42)

,query_id,query_text,is_geospatial_pred
982495,449134,meaning of santa muerte colors,0
278573,575569,what are the wages for pharmacy tech,0
419219,592215,what causes sudden memory,0
534073,840614,what is the price of barley per ton,0
625294,819446,what is the dose of xanax for sleep,0
883351,1167968,weather in dublin ir,1
176763,1062104,why did americans go to mexico?,1
360537,455518,molina healthcare phone number,1
85878,487545,reno weather average by month,1
905953,264847,how long is the flight from las vegas to balti...,1


In [55]:
queries.to_csv('output/queries-classified.20260430.csv.zip', index=False)

In [56]:
final = pd.read_csv('output/queries-classified.20260430.csv.zip')
final

,query_id,query_text,is_geospatial_pred
0,1048578,cost of endless pools/swim spa,0
1,1048579,what is pcnt,0
2,1048580,what is pcb waste,0
3,1048581,what is pbis?,0
4,1048582,what is paysky,0
...,...,...,...
1010911,633855,what does canada post regulations mean,1
1010912,1059728,wholesale lularoe price,0
1010913,210839,how can i watch the day after,0
1010914,908165,what to use instead of pgp in windows,0


In [58]:
final.is_geospatial_pred.value_counts()

is_geospatial_pred
0    828222
1    182694
Name: count, dtype: int64

In [59]:
final.is_geospatial_pred.eq(1).mean()

0.18072124686917607

In [60]:
# What is the most common first word in spatial queries?
top_20_spatial = (final[final.is_geospatial_pred.eq(1)].query_text.str.split(' ').str[0].str.lower().value_counts()
    / final.is_geospatial_pred.eq(1).sum()
     * 100
).head(20).round(1)

top_20_spatial

query_text
what           29.5
where          15.8
how            11.6
average         3.4
when            3.3
weather         2.8
is              2.5
which           1.9
who             1.8
why             1.1
cost            1.0
population      0.9
what's          0.8
does            0.6
can             0.5
most            0.5
largest         0.5
distance        0.5
temperature     0.5
the             0.4
Name: count, dtype: float64

In [61]:
# What about non-spatial queries?
top_20_nonspatial = (final[final.is_geospatial_pred.eq(0)].query_text.str.split(' ').str[0].str.lower().value_counts()
     / final.is_geospatial_pred.eq(0).sum()
     * 100
).head(20).round(1)

top_20_nonspatial

query_text
what          36.2
how           17.9
who            3.7
is             3.0
when           2.6
can            2.1
why            1.8
which          1.8
does           1.3
average        1.2
define         1.0
definition     1.0
cost           1.0
where          0.7
do             0.7
the            0.6
are            0.6
meaning        0.5
what's         0.4
causes         0.4
Name: count, dtype: float64

In [62]:
pd.concat([top_20_spatial, top_20_nonspatial]).index.value_counts()

query_text
what           2
who            2
the            2
can            2
where          2
what's         2
cost           2
why            2
does           2
which          2
how            2
when           2
average        2
is             2
meaning        1
are            1
do             1
definition     1
define         1
distance       1
temperature    1
largest        1
most           1
weather        1
population     1
causes         1
Name: count, dtype: int64